<style>
  .jp-MarkdownCell h1, .text_cell_render h1 { color: #312e81; font-weight: 700; border-bottom: 1px solid #6366f1; padding-bottom: 0.35em; }
  .jp-MarkdownCell h2, .text_cell_render h2 { color: #4338ca; font-weight: 600; margin-top: 1em; border-left: 3px solid #6366f1; padding-left: 0.4em; }
  .jp-MarkdownCell h3, .text_cell_render h3 { color: #0f766e; font-weight: 600; margin-top: 0.9em; border-left: 3px solid #14b8a6; padding-left: 0.35em; }
  .jp-MarkdownCell h4, .text_cell_render h4 { color: #374151; font-weight: 600; margin-top: 0.75em; border-left: 2px solid #94a3b8; padding-left: 0.35em; }
  .jp-MarkdownCell a, .text_cell_render a { color: #4f46e5; }
  .jp-MarkdownCell code, .text_cell_render code { background: #e0e7ff; color: #3730a3; padding: 0.15em 0.4em; border-radius: 4px; font-size: 0.9em; }
  .jp-MarkdownCell pre, .text_cell_render pre { background: #f8fafc; border: 1px solid #e2e8f0; border-left: 3px solid #6366f1; border-radius: 4px; padding: 0.6em 0.8em; }
  .jp-MarkdownCell strong, .text_cell_render strong { color: #1e293b; }
  .jp-MarkdownCell li::marker, .text_cell_render li::marker { color: #6366f1; }
</style>

# 4. Multi-Tenant Vector Databasing with Milvus for RAG and Fine-tuning with GPU/QAIC (Instructor: Mohammad Sada)

The tutorial focuses on NRP-managed Milvus as a multi-tenant vector database service. Participants are guided through accessing Milvus with their credentials, defining collections, and inserting embeddings for semantic search. Using either GPU or QAIC resources, attendees will integrate Milvus into RAG or fine-tuning workflows.

## Milvus on NRP

NRP runs **Milvus** as a managed service, so you don’t install the operator or create the instance yourself. The gRPC endpoint is **`milvus.nrp-nautilus.io:50051`**. To get access, use the [Milvus password page](https://nrp.ai/milvus) and click “Get milvus password”; the secure link is emailed to you. You must be in a [namespace/group with Milvus enabled](https://nrp.ai/namespaces). Group names may use letters, numbers, and dashes; any dashes are converted to underscores in the Milvus database name. From there you can [define collections](https://milvus.io/docs/manage-collections.md), run [vector search](https://milvus.io/docs/single-vector-search.md), or use the [Attu GUI](https://milvus.io/docs/quickstart_with_attu.md). Full reference: [NRP vector database](https://nrp.ai/documentation/userdocs/vector-database/).

## RAG example using Milvus

**Prerequisite:** Get your Milvus password and ensure your group has Milvus enabled (see “Milvus on NRP” above) before starting the pod.

This example demonstrates RAG using Milvus as the vector database. In `yamls/milvus-rag.yaml`, **replace `<username>`** with your username so the pod name `tutorial-<username>-vectordb` is unique. Start up the pod (namespace `nrp-training-k8s`; create it with `kubectl create namespace nrp-training-k8s` if needed):
```
kubectl apply -n nrp-training-k8s -f yamls/milvus-rag.yaml
```

Apply the Milvus RAG pod:

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/milvus-rag.yaml

Watch the logs and make sure you wait till the installs are done:

```
kubectl logs -n nrp-training-k8s tutorial-<username>-vectordb
```
Replace `<username>` with the name you used in the YAML.

In [ ]:
!kubectl logs -n nrp-training-k8s tutorial-<username>-vectordb

The pod automatically installs Python dependencies, Ollama, starts the Ollama server, and downloads the mistral model. Download the example script into the pod once it is running (example repo: [groundsada/nrp-milvus-example](https://github.com/groundsada/nrp-milvus-example)):
```
kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# then inside the pod:
cd /scratch
wget https://raw.githubusercontent.com/groundsada/nrp-milvus-example/refs/heads/main/milvus-example.py
```

In [ ]:
!kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# In the next cell or inside the pod: cd /scratch && wget https://raw.githubusercontent.com/groundsada/nrp-milvus-example/refs/heads/main/milvus-example.py

Once the installation is complete (check logs), you can run the example. The script uses the connection details from the secret `nrp-training-milvus-credentials` (host, port, username, password, database). The collection name is `simple_rag_example`. To use a different collection, edit `milvus-example.py` and change `collection_name="simple_rag_example"` (there are two places where the collection name is set).

Run the script inside the pod (see next cell). The script uses environment variables for the Milvus connection.

For all other aspects, the script uses environment variables for Milvus connection, so no manual editing is needed.

```
kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# then inside the pod:
cd /scratch
python3 milvus-example.py
```

In [ ]:
!kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# Then run: cd /scratch && python3 milvus-example.py

---
## Milvus RAG on Qualcomm Cloud AI 100 Ultra

The RAG workflow above uses an **NVIDIA GPU** pod with Ollama for the LLM step. You can run the same Milvus RAG pipeline with the **LLM on Qualcomm Cloud AI 100 Ultra**: embeddings and Milvus stay on the existing pod; only the generative step uses a vLLM server running on QAIC (as in Part 3 Option B).

**Two ways to run:**

1. **Via JupyterHub** — Use the **NRP Cloud AI Inference** profile (Part 3): spawn with 1 Qualcomm device, then run the QAIC vLLM server and RAG steps from a terminal in that environment. The Milvus RAG pod (from the section above) must be running in the same namespace so you can run the RAG script there with `OPENAI_API_BASE` pointing at the QAIC server (e.g. via port-forward or in-cluster service).

2. **Via kubectl** — Deploy the QAIC vLLM server pod and service, then run the RAG script from the Milvus RAG pod (below).

### Deploy the QAIC vLLM server

In `yamls/qaic-vllm-server.yaml` replace `<username>` in the pod name, then apply. The pod runs an OpenAI-compatible vLLM server (TinyLlama) on port 8000; the service `qaic-vllm-server` lets other pods in `nrp-training-k8s` reach it. Same resource limits as Part 3’s QAIC pod (namespace exception may be required; see [NRP policies](https://nrp.ai/documentation/userdocs/start/policies/)).

```
kubectl apply -n nrp-training-k8s -f yamls/qaic-vllm-server.yaml
```


In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/qaic-vllm-server.yaml

Check pod status and logs until the vLLM server is ready:

```
kubectl get pods -n nrp-training-k8s -l app=qaic-vllm-server
kubectl logs -n nrp-training-k8s tutorial-<username>-qaic-vllm
```

Then run the Milvus RAG script **from the Milvus RAG pod**, pointing the LLM at the QAIC server:

```
kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- bash -c 'cd /scratch && wget -q https://raw.githubusercontent.com/nrp-nautilus/nairr-tutorial/main/scripts/milvus_rag_qaic.py && OPENAI_API_BASE=http://qaic-vllm-server:8000/v1 python3 milvus_rag_qaic.py'
```

Replace `<username>` in the pod name with the value you used in the YAMLs. The script uses the same Milvus credentials and collection; only the generative step uses the Qualcomm vLLM server.

In [ ]:
!kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- bash -c 'cd /scratch && wget -q https://raw.githubusercontent.com/nrp-nautilus/nairr-tutorial/main/scripts/milvus_rag_qaic.py && OPENAI_API_BASE=http://qaic-vllm-server:8000/v1 python3 milvus_rag_qaic.py'

This example uses a small set of sample documents to demonstrate Milvus vector storage and retrieval and RAG with Ollama (NVIDIA) or with the Qualcomm vLLM server.

## End — cleanup

Delete the Milvus RAG pod and, if you ran the Qualcomm section, the QAIC vLLM server (replace `<username>` with the name you used in the YAMLs):

```bash
kubectl delete -n nrp-training-k8s -f yamls/milvus-rag.yaml --ignore-not-found
kubectl delete -n nrp-training-k8s -f yamls/qaic-vllm-server.yaml --ignore-not-found
# or: kubectl delete pod -n nrp-training-k8s tutorial-<username>-vectordb tutorial-<username>-qaic-vllm
```

**Please make sure you did not leave any other running pods. Jobs and associated completed pods are OK.**

**Join NRP & hands-on support:** Use these links to [get started with NRP](https://nrp.ai/documentation/userdocs/start/getting-started/) and to [join NRP's Matrix chat](https://nrp.ai/contact/) for hands-on support during the tutorial.

**Related documentation:** [NRP-managed vector database (Milvus)](https://nrp.ai/documentation/userdocs/vector-database/) · [NRP-managed LLMs](https://nrp.ai/documentation/userdocs/ai/llm-managed/) · [Qualcomm Cloud AI 100](https://nrp.ai/documentation/userdocs/qaic/)